In [3]:
import os
import json
import numpy as np
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F

In [4]:
DATA_DIR = "./data"
CSV_DIR = f"{DATA_DIR}/csv"

SONGS_PATH = f"{CSV_DIR}/songs.csv"
TAGS_PATH  = f"{CSV_DIR}/tags.csv"

OUTPUT_PATH = f"{DATA_DIR}/embeddings/song_token_embeddings.json"

EMBED_DIM = 128
WINDOW = 3
NEG_SAMPLES = 8
BATCH_SIZE = 2048
EPOCHS = 30
LR = 1e-3

In [5]:
def read_csv(path):
    with open(path, encoding="utf-8") as f:
        lines = f.read().strip().splitlines()

    headers = [h.strip('"') for h in lines[0].split(",")]
    rows = []

    for line in lines[1:]:
        parts = [p.strip('"') for p in line.split(";")]
        if len(parts) == len(headers):
            rows.append(dict(zip(headers, parts)))

    return rows

In [6]:
docs = defaultdict(set)

# --- genres ---
for row in read_csv(SONGS_PATH):
    sid = row["spotify_id"]
    genre = row["genre_name"].lower().strip()

    docs[sid].add(f"g_{genre}")   # prefix = important

# --- tags (no popularity weighting) ---
for row in read_csv(TAGS_PATH):
    sid = row["song_spotify_id"]
    tag = row["tag"].lower().strip()

    docs[sid].add(f"t_{tag}")     # prefix separates space

In [15]:
docs

defaultdict(set,
            {'6lV2MSQmRIkycDScNtrBXO': {'g_dance pop',
              'g_dwn trap',
              'g_pop',
              'g_pop rap',
              'g_rap',
              'g_southern hip hop',
              'g_trap music'},
             '4hrae8atte6cRlSC9a7VCO': {'g_crunk',
              'g_dirty south rap',
              'g_east coast hip hop',
              'g_gangster rap',
              'g_hip hop',
              'g_hip pop',
              'g_pop rap',
              'g_r&b',
              'g_rap',
              'g_urban contemporary',
              't_00s',
              't_1',
              't_2000s',
              't_2002',
              't_3',
              't_always on time',
              't_always on time - ja rule',
              't_alwayz on time',
              't_american',
              't_andy',
              't_awesome',
              't_baby',
              't_bergen',
              't_black',
              't_catchy',
              't_chill',
        

In [7]:
documents = [list(tokens) for tokens in docs.values() if len(tokens) > 1]

print("Songs:", len(documents))
print("Example:", documents[0])

Songs: 81244
Example: ['g_rap', 'g_dance pop', 'g_southern hip hop', 'g_pop rap', 'g_pop', 'g_dwn trap', 'g_trap music']


In [16]:
documents

[['g_rap',
  'g_dance pop',
  'g_southern hip hop',
  'g_pop rap',
  'g_pop',
  'g_dwn trap',
  'g_trap music'],
 ['g_dirty south rap',
  't_ja rule',
  't_ja rule always on time',
  't_pozytywne wibracje',
  'g_gangster rap',
  't_urban',
  't_hits',
  't_00s',
  't_loved',
  't_sweet',
  't_combo',
  't_mixed vocals',
  't_3',
  't_favourite',
  't_summer tunes',
  't_my parties',
  't_rap',
  't_power',
  't_popular rap',
  't_funk',
  't_love',
  't_loneliness after dusk',
  't_download',
  'g_crunk',
  'g_r&b',
  'g_hip pop',
  't_awesome',
  't_gangsta rap',
  't_sick',
  't_hip hop',
  't_good',
  't_slow jams',
  't_alwayz on time',
  't_female vocal',
  'g_rap',
  't_dancemania',
  't_good shit',
  't_hardcore rap',
  't_soul',
  't_melhores da black music',
  't_house',
  't_duets',
  't_margaret',
  'g_urban contemporary',
  't_nelly',
  'g_hip hop',
  't_mainstream',
  't_chill',
  't_personal favourites',
  't_andy',
  't_jarule',
  't_1',
  't_female vocalists',
  't_ja r

In [8]:
freq = defaultdict(int)

for doc in documents:
    for t in doc:
        freq[t] += 1

vocab = list(freq.keys())
token2idx = {t:i for i,t in enumerate(vocab)}
idx2token = {i:t for t,i in token2idx.items()}

V = len(vocab)
print("Vocab size:", V)

Vocab size: 156610


In [20]:
idx2token

{0: 'g_rap',
 1: 'g_dance pop',
 2: 'g_southern hip hop',
 3: 'g_pop rap',
 4: 'g_pop',
 5: 'g_dwn trap',
 6: 'g_trap music',
 7: 'g_dirty south rap',
 8: 't_ja rule',
 9: 't_ja rule always on time',
 10: 't_pozytywne wibracje',
 11: 'g_gangster rap',
 12: 't_urban',
 13: 't_hits',
 14: 't_00s',
 15: 't_loved',
 16: 't_sweet',
 17: 't_combo',
 18: 't_mixed vocals',
 19: 't_3',
 20: 't_favourite',
 21: 't_summer tunes',
 22: 't_my parties',
 23: 't_rap',
 24: 't_power',
 25: 't_popular rap',
 26: 't_funk',
 27: 't_love',
 28: 't_loneliness after dusk',
 29: 't_download',
 30: 'g_crunk',
 31: 'g_r&b',
 32: 'g_hip pop',
 33: 't_awesome',
 34: 't_gangsta rap',
 35: 't_sick',
 36: 't_hip hop',
 37: 't_good',
 38: 't_slow jams',
 39: 't_alwayz on time',
 40: 't_female vocal',
 41: 't_dancemania',
 42: 't_good shit',
 43: 't_hardcore rap',
 44: 't_soul',
 45: 't_melhores da black music',
 46: 't_house',
 47: 't_duets',
 48: 't_margaret',
 49: 'g_urban contemporary',
 50: 't_nelly',
 51: 'g_hi

In [19]:
token2idx

{'g_rap': 0,
 'g_dance pop': 1,
 'g_southern hip hop': 2,
 'g_pop rap': 3,
 'g_pop': 4,
 'g_dwn trap': 5,
 'g_trap music': 6,
 'g_dirty south rap': 7,
 't_ja rule': 8,
 't_ja rule always on time': 9,
 't_pozytywne wibracje': 10,
 'g_gangster rap': 11,
 't_urban': 12,
 't_hits': 13,
 't_00s': 14,
 't_loved': 15,
 't_sweet': 16,
 't_combo': 17,
 't_mixed vocals': 18,
 't_3': 19,
 't_favourite': 20,
 't_summer tunes': 21,
 't_my parties': 22,
 't_rap': 23,
 't_power': 24,
 't_popular rap': 25,
 't_funk': 26,
 't_love': 27,
 't_loneliness after dusk': 28,
 't_download': 29,
 'g_crunk': 30,
 'g_r&b': 31,
 'g_hip pop': 32,
 't_awesome': 33,
 't_gangsta rap': 34,
 't_sick': 35,
 't_hip hop': 36,
 't_good': 37,
 't_slow jams': 38,
 't_alwayz on time': 39,
 't_female vocal': 40,
 't_dancemania': 41,
 't_good shit': 42,
 't_hardcore rap': 43,
 't_soul': 44,
 't_melhores da black music': 45,
 't_house': 46,
 't_duets': 47,
 't_margaret': 48,
 'g_urban contemporary': 49,
 't_nelly': 50,
 'g_hip ho

In [18]:
vocab

['g_rap',
 'g_dance pop',
 'g_southern hip hop',
 'g_pop rap',
 'g_pop',
 'g_dwn trap',
 'g_trap music',
 'g_dirty south rap',
 't_ja rule',
 't_ja rule always on time',
 't_pozytywne wibracje',
 'g_gangster rap',
 't_urban',
 't_hits',
 't_00s',
 't_loved',
 't_sweet',
 't_combo',
 't_mixed vocals',
 't_3',
 't_favourite',
 't_summer tunes',
 't_my parties',
 't_rap',
 't_power',
 't_popular rap',
 't_funk',
 't_love',
 't_loneliness after dusk',
 't_download',
 'g_crunk',
 'g_r&b',
 'g_hip pop',
 't_awesome',
 't_gangsta rap',
 't_sick',
 't_hip hop',
 't_good',
 't_slow jams',
 't_alwayz on time',
 't_female vocal',
 't_dancemania',
 't_good shit',
 't_hardcore rap',
 't_soul',
 't_melhores da black music',
 't_house',
 't_duets',
 't_margaret',
 'g_urban contemporary',
 't_nelly',
 'g_hip hop',
 't_mainstream',
 't_chill',
 't_personal favourites',
 't_andy',
 't_jarule',
 't_1',
 't_female vocalists',
 't_ja rule - always on time',
 't_male rappers',
 't_featuring',
 't_mine',
 't

In [25]:
freq

defaultdict(int,
            {'g_rap': 191,
             'g_dance pop': 196,
             'g_southern hip hop': 199,
             'g_pop rap': 198,
             'g_pop': 199,
             'g_dwn trap': 136,
             'g_trap music': 197,
             'g_dirty south rap': 196,
             't_ja rule': 2,
             't_ja rule always on time': 1,
             't_pozytywne wibracje': 86,
             'g_gangster rap': 194,
             't_urban': 339,
             't_hits': 165,
             't_00s': 2597,
             't_loved': 1699,
             't_sweet': 905,
             't_combo': 3,
             't_mixed vocals': 2,
             't_3': 454,
             't_favourite': 1614,
             't_summer tunes': 4,
             't_my parties': 29,
             't_rap': 1203,
             't_power': 191,
             't_popular rap': 1,
             't_funk': 1576,
             't_love': 4384,
             't_loneliness after dusk': 99,
             't_download': 398,
             'g

In [26]:
pairs = []

for doc in documents:
    idxs = [token2idx[t] for t in doc]

    for i in range(len(idxs)):
        for j in range(len(idxs)):
            if i != j:
                pairs.append((idxs[i], idxs[j]))

pairs = np.array(pairs, dtype=np.int32)
print("Pairs:", len(pairs))

Pairs: 61189554


In [33]:
pairs[140:170]

array([[  7, 105],
       [  7, 106],
       [  7, 107],
       [  7, 108],
       [  7, 109],
       [  7, 110],
       [  7,   3],
       [  7, 111],
       [  7, 112],
       [  7, 113],
       [  7, 114],
       [  8,   7],
       [  8,   9],
       [  8,  10],
       [  8,  11],
       [  8,  12],
       [  8,  13],
       [  8,  14],
       [  8,  15],
       [  8,  16],
       [  8,  17],
       [  8,  18],
       [  8,  19],
       [  8,  20],
       [  8,  21],
       [  8,  22],
       [  8,  23],
       [  8,  24],
       [  8,  25],
       [  8,  26]], dtype=int32)

In [34]:
import torch
import torch.nn as nn

class SGNS(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        # input embedding (center word)
        self.in_embed = nn.Embedding(vocab_size, embed_dim)

        # output embedding (context word)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

        # init (important for stability)
        nn.init.uniform_(self.in_embed.weight, -0.1, 0.1)
        nn.init.zeros_(self.out_embed.weight)

    def forward(self, center, context, negatives):
        """
        center:   (B,)
        context:  (B,)
        negatives:(B, K)
        """

        v_c = self.in_embed(center)        # (B, D)
        v_o = self.out_embed(context)      # (B, D)
        v_n = self.out_embed(negatives)    # (B, K, D)

        # positive score: center · context
        pos_score = (v_c * v_o).sum(dim=1)
        pos_loss = torch.log(torch.sigmoid(pos_score + 1e-9))

        # negative score: center · negative samples
        neg_score = torch.bmm(
            v_n, v_c.unsqueeze(2)
        ).squeeze(2)

        neg_loss = torch.log(torch.sigmoid(-neg_score + 1e-9)).sum(dim=1)

        # final loss
        return -(pos_loss + neg_loss).mean()

In [35]:
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SGNS(V, EMBED_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

pairs_tensor = torch.tensor(pairs, dtype=torch.long)

NEG_K = 8
BATCH_SIZE = 2048

for epoch in range(20):
    perm = torch.randperm(len(pairs_tensor))
    shuffled = pairs_tensor[perm]

    total_loss = 0

    for i in range(0, len(shuffled), BATCH_SIZE):
        batch = shuffled[i:i+BATCH_SIZE].to(device)

        center = batch[:,0]
        context = batch[:,1]

        negatives = torch.randint(0, V, (len(batch), NEG_K), device=device)

        loss = model(center, context, negatives)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"epoch {epoch+1} loss: {total_loss:.4f}")

epoch 1 loss: 40083.4837
epoch 2 loss: 26813.9767
epoch 3 loss: 24083.3536
epoch 4 loss: 23012.7480
epoch 5 loss: 22492.1205
epoch 6 loss: 22185.8941
epoch 7 loss: 21980.2288
epoch 8 loss: 21835.2060
epoch 9 loss: 21732.4192
epoch 10 loss: 21647.8749
epoch 11 loss: 21577.0400
epoch 12 loss: 21520.4424
epoch 13 loss: 21462.8466
epoch 14 loss: 21419.6188
epoch 15 loss: 21378.5072
epoch 16 loss: 21344.0079
epoch 17 loss: 21319.2036
epoch 18 loss: 21298.5765
epoch 19 loss: 21275.5129
epoch 20 loss: 21260.9048


In [36]:
# after training loop, add this:
model.eval()
embeddings = {}

with torch.no_grad():
    for token, idx in token2idx.items():
        vec = model.in_embed.weight[idx].cpu().numpy()
        embeddings[token] = vec.tolist()

with open(OUTPUT_PATH, "w") as f:
    json.dump(embeddings, f)

print(f"Saved {len(embeddings)} embeddings to {OUTPUT_PATH}")

Saved 156610 embeddings to ./data/embeddings/song_token_embeddings.json


In [37]:
# after saving embeddings, quick neighbor check
with open(OUTPUT_PATH) as f:
    raw = json.load(f)

tokens  = list(raw.keys())
token2i = {t: i for i, t in enumerate(tokens)}
E       = np.array(list(raw.values()), dtype=np.float32)
E_norm  = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-9)

def neighbors(query, k=5):
    if query not in token2i:
        print(f"{query} not in vocab"); return
    q    = E_norm[token2i[query]]
    sims = E_norm @ q
    top  = np.argsort(sims)[::-1][1:k+1]
    print(f"{query:30s} → {[tokens[i] for i in top]}")

neighbors("g_pop rap")
neighbors("g_death metal")
neighbors("g_jazz")
neighbors("g_classical")

g_pop rap                      → ['g_southern hip hop', 'g_rap', 'g_trap music', 'g_dirty south rap', 'g_crunk']
g_death metal                  → ['g_technical death metal', 'g_brutal death metal', 'g_thrash metal', 'g_deathgrind', 't_fuk u fuk u fuk u fuk u fuk u fuk u fuk u fuk u']
g_jazz                         → ['g_bebop', 'g_cool jazz', 'g_contemporary post-bop', 'g_hard bop', 'g_soul jazz']
g_classical                    → ['t_tom and jerry', 't_well known classical', 't_classical awesome', 'g_romantic', 't_george gershwin - rhapsody in blue']


In [38]:
# filter to just genre tokens for VAE input
with open(OUTPUT_PATH) as f:
    raw = json.load(f)

genre_embeddings = {
    k: v for k, v in raw.items() 
    if k.startswith("g_")
}

print(f"genre tokens: {len(genre_embeddings)}")  # should be much smaller

# save separately for VAE
with open(f"{DATA_DIR}/embeddings/genre_only_embeddings.json", "w") as f:
    json.dump(genre_embeddings, f)

genre tokens: 1493
